In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import seaborn as sns
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv, SubprocVecEnv

import sumo_rl_ego as sre
from human_feedback_rl.common.trajectory_generators import TrajectoryGeneratorFromAgent

from loadings import load_reward_ensemble
from plot_functions import plot_scatter, plot_reward_curves

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams["figure.dpi"] = 120

Plot functions loaded OK


In [2]:
CHECKPOINT_DIR = Path("/Users/andreazhang/Documents/fis/sumo-human-feedback-rl/outputs/seg1_fragrandom_comps2000_seed10/checkpoint_0150")

import sumo_gym_ego.core.simulation as _sim_mod
import traci as _traci_mod
_sim_mod.load_traci = lambda use_gui: _traci_mod

try:
    env.close()
except Exception:
    pass

env = sre.make_vec_env(
    "HighwayEgo-v0",
    n_envs=1,
    base_seed=0,
    ego="continuous",
    reward="fast"
)

reward_model = load_reward_ensemble(
    CHECKPOINT_DIR / "reward_model.pt",
    env.observation_space,
    env.action_space,
)

# Passa env a load() per fare l'override di n_envs (il modello era salvato con n_envs=4)
agent = PPO.load(CHECKPOINT_DIR / "agent.zip", env=env, device="cpu")

trajectory_generator = TrajectoryGeneratorFromAgent(
    agent=agent,
    reward_model=reward_model,
    venv=env,
)

In [3]:
N_STEPS = 100000
trajectories, log_metrics = trajectory_generator.sample(N_STEPS)

 Retrying in 1 seconds


In [17]:
N_STEPS = 30000
trajectories2, log_metrics = trajectory_generator.sample(N_STEPS)
trajectories.extend(trajectories2)

In [18]:
import numpy as np


# next_status: 7-dim one-hot [arrived, collided, off_road, timeout, running, teleported, removed_unknown]
STATUS_ARRIVED  = 0
STATUS_COLLIDED = 1
STATUS_OFFROAD  = 2
STATUS_TIMEOUT  = 3

terminal_statuses = [np.argmax(traj[-1].next_status) for traj in trajectories]

n_episodes = len(trajectories)
n_arrived  = sum(s == STATUS_ARRIVED  for s in terminal_statuses)
n_collided = sum(s == STATUS_COLLIDED for s in terminal_statuses)
n_offroad  = sum(s == STATUS_OFFROAD  for s in terminal_statuses)
n_timeout  = sum(s == STATUS_TIMEOUT  for s in terminal_statuses)

print(f"Episodi totali:        {n_episodes}")
print(f"Lunghezza media:       {log_metrics.mean_length:.1f}")
print(f"Reward media:          {log_metrics.mean_true_reward:.3f}")
print(f"Reward predetta media: {log_metrics.mean_model_reward:.3f}")
print()
print(f"Terminazioni:")
print(f"  Arrived:  {n_arrived:3d}  ({100 * n_arrived  / n_episodes:.1f}%)")
print(f"  Collided: {n_collided:3d}  ({100 * n_collided / n_episodes:.1f}%)")
print(f"  Off-road: {n_offroad:3d}  ({100 * n_offroad  / n_episodes:.1f}%)")
print(f"  Timeout:  {n_timeout:3d}  ({100 * n_timeout  / n_episodes:.1f}%)")

Episodi totali:        464
Lunghezza media:       357.2
Reward media:          -28.734
Reward predetta media: 250.559

Terminazioni:
  Arrived:   41  (8.8%)
  Collided: 198  (42.7%)
  Off-road:  25  (5.4%)
  Timeout:  200  (43.1%)


In [30]:
N_PER_CLASS = 20

buckets = {
    STATUS_ARRIVED:  [],
    STATUS_COLLIDED: [],
    STATUS_OFFROAD:  [],
    STATUS_TIMEOUT:  [],
}

for traj, status in zip(trajectories, terminal_statuses):
    if status in buckets and len(buckets[status]) < N_PER_CLASS:
        buckets[status].append(traj)

trajectories_balanced = (
    buckets[STATUS_ARRIVED] +
    buckets[STATUS_COLLIDED] +
    buckets[STATUS_OFFROAD] +
    buckets[STATUS_TIMEOUT]
)

print(f"Traiettorie bilanciate: {len(trajectories_balanced)}")
print(f"  Arrived:  {len(buckets[STATUS_ARRIVED])}")
print(f"  Collided: {len(buckets[STATUS_COLLIDED])}")
print(f"  Off-road: {len(buckets[STATUS_OFFROAD])}")
print(f"  Timeout:  {len(buckets[STATUS_TIMEOUT])}")

Traiettorie bilanciate: 80
  Arrived:  20
  Collided: 20
  Off-road: 20
  Timeout:  20


In [29]:
STATUS_RUNNING = 4
L = 10000  # lunghezza segmento — modifica qui

segment_buckets = {
    STATUS_ARRIVED:  [],
    STATUS_COLLIDED: [],
    STATUS_OFFROAD:  [],
    STATUS_TIMEOUT:  [],
    STATUS_RUNNING:  [],
}

for traj, term_status in zip(trajectories, terminal_statuses):
    if len(traj) < L:
        continue

    # segmenti terminali: ultime L transizioni di traiettorie con lo stato finale atteso
    if term_status in segment_buckets and len(segment_buckets[term_status]) < N_PER_CLASS:
        segment_buckets[term_status].append(traj[-L:])

    # segmenti running: qualsiasi finestra di L passi con tutti i next_status == running
    if len(segment_buckets[STATUS_RUNNING]) < N_PER_CLASS:
        statuses_in_traj = [np.argmax(t.next_status) for t in traj]
        for i in range(len(traj) - L + 1):
            window_statuses = statuses_in_traj[i : i + L]
            if all(s == STATUS_RUNNING for s in window_statuses):
                segment_buckets[STATUS_RUNNING].append(traj[i : i + L])
                break  # un segmento per traiettoria, evita duplicati quasi identici

print(f"Segmenti per classe (L={L}):")
for status, name in [
    (STATUS_ARRIVED,  "Arrived"),
    (STATUS_COLLIDED, "Collided"),
    (STATUS_OFFROAD,  "Off-road"),
    (STATUS_TIMEOUT,  "Timeout"),
    (STATUS_RUNNING,  "Running"),
]:
    print(f"  {name:8s}: {len(segment_buckets[status])}")

Segmenti per classe (L=10000):
  Arrived : 0
  Collided: 0
  Off-road: 0
  Timeout : 0
  Running : 0


In [31]:
import pickle

SAVE_PATH = "debug_dataset_full_ep.pkl"

with open(SAVE_PATH, "wb") as f:
    pickle.dump(buckets, f)

print(f"Salvato in: {SAVE_PATH}")

# Per ricaricare in un altro script:
# with open(SAVE_PATH, "rb") as f:
#     data = pickle.load(f)
# L = data["L"]
# segment_buckets = data["segment_buckets"]

Salvato in: debug_dataset_full_ep.pkl


In [33]:
with open(SAVE_PATH, "rb") as f:
     data = pickle.load(f)


print(f"Segmenti per classe (L={L}):")
for status, name in [
    (STATUS_ARRIVED,  "Arrived"),
    (STATUS_COLLIDED, "Collided"),
    (STATUS_OFFROAD,  "Off-road"),
    (STATUS_TIMEOUT,  "Timeout"),
]:
    print(f"  {name:8s}: {len(data[status])}")

Segmenti per classe (L=10000):
  Arrived : 20
  Collided: 20
  Off-road: 20
  Timeout : 20
